# Trading Intelligent — CH02 — Notebook 1
## Stationnarité, ACF/PACF et diagnostic de dépendance

**Auteur académique : Tawfik Masrour**  
**Version : 2.2**  
**Dataset : `TI_CH02_TS_MODELS_V1`**

## Question directrice
Comment distinguer une véritable dépendance temporelle d'une tendance, d'une racine unitaire ou d'un simple bruit d'échantillonnage ?

## Ce que vous allez apprendre
1. distinguer visuellement et statistiquement une série stationnaire d'une série intégrée ;
2. lire ACF et PACF sans les transformer en « empreintes magiques » ;
3. comprendre l'effet du signe et de l'amplitude de $\phi$ dans un AR(1) ;
4. comprendre pourquoi différencier transforme la dynamique ;
5. interpréter ADF et KPSS sans inverser leurs hypothèses nulles ;
6. voir concrètement pourquoi la spécification `c` versus `ct` peut changer une conclusion.

## Prérequis
Rendements et séries temporelles (CH01), moyenne, variance, corrélation, notions élémentaires de Python/pandas.

## Comment utiliser ce notebook
Exécutez les cellules dans l'ordre. Après chaque graphique, formulez d'abord une interprétation qualitative. Les bandes ACF/PACF sont des repères approximatifs ; les tests complètent le raisonnement sans remplacer le mécanisme.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

SEED = 20260930
DATASET_ID = "TI_CH02_TS_MODELS_V1"
DATASET_FILENAME = "TI_CH02_TS_Models_v1.csv"

# Palette canonique Trading Intelligent
ACADEMIC_BLUE = "#173B57"
PETROL = "#2C7A7B"
WARM_GOLD = "#C79A3B"
MUTED_RED = "#A94B4B"
NEUTRAL_GRAY = "#66717D"

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.edgecolor": NEUTRAL_GRAY,
    "xtick.color": NEUTRAL_GRAY,
    "ytick.color": NEUTRAL_GRAY,
})


def generate_dataset(seed=SEED, n=520):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2024-01-02", periods=n)
    eps = rng.normal(0, 1, n)

    def ar1(phi):
        x = np.zeros(n)
        for t in range(1, n):
            x[t] = phi * x[t-1] + eps[t]
        return x

    def ma1(theta):
        x = eps.copy()
        x[1:] += theta * eps[:-1]
        return x

    random_walk = 100 + np.cumsum(0.12 + rng.normal(0, 1.1, n))
    innov = rng.normal(0, 0.8, n)
    inc = np.zeros(n)
    for t in range(1, n):
        inc[t] = 0.35 * inc[t-1] + innov[t] + 0.45 * innov[t-1] + 0.04
    arima_price = 100 + np.cumsum(inc)

    return pd.DataFrame({
        "date": dates,
        "white_noise": eps,
        "ar1_phi_075": ar1(0.75),
        "ar1_phi_neg_065": ar1(-0.65),
        "ma1_theta_070": ma1(0.70),
        "random_walk_price": random_walk,
        "arima_price": arima_price,
    })


def locate_dataset(filename=DATASET_FILENAME):
    here = Path.cwd()
    roots = [here, here.parent, here.parent.parent]
    for root in roots:
        for sub in [Path('.'), Path('datasets'), Path('data'), Path('resources')]:
            p = (root / sub / filename).resolve()
            if p.exists():
                return p
    return None


path = locate_dataset()
if path is not None:
    df = pd.read_csv(path, parse_dates=['date'])
    print(f"{DATASET_ID} chargé : {len(df)} lignes, {len(df.columns)} colonnes")
else:
    df = generate_dataset()
    print(f"{DATASET_ID} absent localement : version déterministe régénérée en mémoire.")

assert len(df) == 520
assert not df.isna().any().any()
df.head()


In [ ]:
def format_small_p(p):
    """Affichage scientifique : une p-value numérique très petite n'est pas zéro."""
    if p < 1e-10:
        return f"p = {p:.2e} (< 1e-10)"
    if p < 1e-4:
        return f"p = {p:.2e}"
    return f"p = {p:.4f}"


def kpss_with_bound(x, regression="c"):
    """Retourne KPSS en explicitant les bornes de table de Statsmodels."""
    x = pd.Series(x).dropna()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        stat, p, lags, crit = kpss(x, regression=regression, nlags="auto")
    messages = [str(w.message) for w in caught]
    joined = " ".join(messages).lower()
    if "smaller than the p-value returned" in joined:
        p_text = f"p ≤ {p:.2f} (borne de table Statsmodels)"
    elif "greater than the p-value returned" in joined:
        p_text = f"p ≥ {p:.2f} (borne de table Statsmodels)"
    else:
        p_text = format_small_p(p)
    return {"stat": stat, "p": p, "p_text": p_text, "lags": lags, "warnings": messages}


def stationarity_report(x, name, adf_reg="c", kpss_reg="c"):
    x = pd.Series(x).dropna()
    adf = adfuller(x, regression=adf_reg, autolag="AIC")
    kp = kpss_with_bound(x, regression=kpss_reg)
    return pd.Series({
        "ADF_stat": adf[0],
        "ADF_p": adf[1],
        "ADF_p_affichage": format_small_p(adf[1]),
        "ADF_lags": adf[2],
        "KPSS_stat": kp["stat"],
        "KPSS_p": kp["p"],
        "KPSS_p_affichage": kp["p_text"],
        "spec": f"ADF:{adf_reg} / KPSS:{kpss_reg}",
    }, name=name)


In [ ]:
def show_acf_pacf(x, nlags=24, title=""):
    x = pd.Series(x).dropna()
    a = acf(x, nlags=nlags, fft=True)
    p = pacf(x, nlags=nlags, method="ywm")
    bound = 1.96 / np.sqrt(len(x))
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for ax, vals, label, color in [
        (axs[0], a, "ACF", ACADEMIC_BLUE),
        (axs[1], p, "PACF", PETROL),
    ]:
        markerline, stemlines, baseline = ax.stem(range(len(vals)), vals, basefmt=" ")
        plt.setp(stemlines, color=color)
        plt.setp(markerline, color=color)
        ax.axhline(bound, ls="--", lw=0.9, color=WARM_GOLD)
        ax.axhline(-bound, ls="--", lw=0.9, color=WARM_GOLD)
        ax.axhline(0, lw=0.7, color=NEUTRAL_GRAY)
        ax.grid(alpha=0.2)
        ax.set_xlabel("Retard")
        ax.set_title(title + " — " + label)
    plt.tight_layout(); plt.show()
    print(f"Bandes approximatives : ±1,96/√N = ±{bound:.3f}. Repère pédagogique, pas règle automatique.")


## 1. Contrôle du jeu de données

Le dataset est synthétique : nous connaissons le mécanisme générateur. C'est volontaire. Avant d'analyser des marchés réels, nous vérifions que nos outils reconnaissent des structures dont la vérité est connue.


In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
summary


## 2. Bruit blanc, stationnarité et marche aléatoire

Un bruit blanc faible est non autocorrélé, mais cela ne signifie pas en général que ses observations sont indépendantes ; il n'est pas non plus nécessairement gaussien. Une marche aléatoire, elle, accumule les chocs : un choc modifie durablement son niveau.


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axs[0].plot(df.date, df.ar1_phi_075, color=ACADEMIC_BLUE)
axs[0].axhline(0, lw=0.8, color=NEUTRAL_GRAY)
axs[0].set_title("AR(1), φ = 0,75 — fluctuations autour d'un niveau stable")
axs[0].grid(alpha=0.2)
axs[1].plot(df.date, df.random_walk_price, color=PETROL)
axs[1].set_title("Marche aléatoire — le niveau dérive avec les chocs cumulés")
axs[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()


### Question rapide
Si l'on calcule l'autocorrélation du **niveau** d'une marche aléatoire, doit-on interpréter une valeur élevée comme une preuve de prédictibilité économique ? Justifiez avant de continuer.


## 3. ACF et PACF : diagnostic, pas reconnaissance automatique

Pour un AR($p$) idéal, la PACF tend à se couper après $p$ ; pour un MA($q$) idéal, c'est l'ACF qui tend à se couper après $q$. Sur un échantillon fini, bruit d'échantillonnage, saisonnalité, ruptures, paramètres proches des frontières et hétéroscédasticité peuvent brouiller les signatures.

Les lignes pointillées $\pm 1{,}96/\sqrt{N}$ sont uniquement un repère pédagogique approximatif.


In [ ]:
show_acf_pacf(df.ar1_phi_075, title="AR(1), φ = 0,75")
show_acf_pacf(df.ma1_theta_070, title="MA(1), θ = 0,70")


## 4. AR(1) : isoler l'effet de $\phi$ avec les mêmes innovations

Comparer des trajectoires générées avec des bruits différents mélange deux effets. Ici, les trois processus utilisent **exactement la même réalisation des innovations** afin d'isoler visuellement l'effet de $\phi$.


In [ ]:
rng_same = np.random.default_rng(SEED + 4)
eps_same = rng_same.normal(size=180)

def ar1_from_eps(phi, eps):
    x = np.zeros(len(eps))
    for t in range(1, len(eps)):
        x[t] = phi * x[t-1] + eps[t]
    return x

fig, ax = plt.subplots(figsize=(11, 4.5))
for phi, color in [(0.2, NEUTRAL_GRAY), (0.7, PETROL), (0.95, ACADEMIC_BLUE)]:
    ax.plot(ar1_from_eps(phi, eps_same), label=f"φ={phi}", color=color)
ax.set_title("AR(1) — mêmes innovations, persistance différente")
ax.legend(); ax.grid(alpha=0.2); plt.show()


## 5. Différencier : utile, mais jamais neutre

La première différence d'une marche aléatoire retrouve essentiellement ses innovations. Mais si une série est déjà stationnaire, la différenciation peut introduire une structure de type MA artificielle.


In [ ]:
rw = df.random_walk_price
rw_diff1 = rw.diff().dropna()
stationary = pd.Series(df.ar1_phi_075)
stationary_diff = stationary.diff().dropna()

fig, axs = plt.subplots(2, 1, figsize=(11, 7))
axs[0].plot(df.date.iloc[1:], rw_diff1, color=PETROL)
axs[0].set_title("Marche aléatoire différenciée une fois")
axs[0].grid(alpha=0.2)
axs[1].plot(df.date.iloc[1:], stationary_diff, color=MUTED_RED)
axs[1].set_title("AR(1) déjà stationnaire puis différencié : dynamique transformée")
axs[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()
show_acf_pacf(stationary_diff, title="Δ d'un AR(1) déjà stationnaire")


## 6. ADF + KPSS : deux hypothèses nulles différentes

- **ADF** : $H_0$ = racine unitaire.
- **KPSS** : $H_0$ = stationnarité autour d'un niveau ou d'une tendance selon la spécification.

Les p-values KPSS de Statsmodels sont parfois des **bornes de table**. Le notebook affiche alors `p ≤ 0,01` ou `p ≥ 0,10`, et non une fausse valeur exacte. De même, une p-value ADF extrêmement petite n'est jamais affichée comme `0.0`.


In [ ]:
report = pd.concat([
    stationarity_report(df.ar1_phi_075, "AR(1)"),
    stationarity_report(df.random_walk_price, "Marche aléatoire"),
    stationarity_report(df.random_walk_price.diff(), "Différence RW"),
    stationarity_report(df.white_noise, "Bruit blanc"),
], axis=1).T
report[["ADF_stat","ADF_p_affichage","ADF_lags","KPSS_stat","KPSS_p_affichage","spec"]]


## 7. Expérience clé : série stationnaire autour d'une tendance

Nous simulons
$$Y_t=a+bt+u_t,$$
avec $u_t$ stationnaire. La série possède une tendance déterministe ; elle n'est pas une marche aléatoire. Nous posons successivement deux questions différentes : stationnarité autour d'un **niveau** (`c`) puis autour d'une **tendance** (`ct`).


In [ ]:
rng_trend = np.random.default_rng(SEED + 27)
n_trend = 240
t = np.arange(n_trend)
e = rng_trend.normal(size=n_trend)
u = np.zeros(n_trend)
for i in range(1, n_trend):
    u[i] = 0.55 * u[i-1] + e[i]
trend_stationary = 100 + 0.08*t + u

fig, ax = plt.subplots(figsize=(11,4.5))
ax.plot(t, trend_stationary, color=ACADEMIC_BLUE, label="Série simulée")
ax.plot(t, 100 + 0.08*t, ls="--", color=WARM_GOLD, label="Tendance déterministe")
ax.set_title("Trend-stationary : fluctuations stationnaires autour d'une tendance")
ax.legend(); ax.grid(alpha=0.2); plt.show()

spec_demo = pd.concat([
    stationarity_report(trend_stationary, "constante seulement", adf_reg="c", kpss_reg="c"),
    stationarity_report(trend_stationary, "constante + tendance", adf_reg="ct", kpss_reg="ct"),
], axis=1).T
spec_demo[["ADF_stat","ADF_p_affichage","ADF_lags","KPSS_stat","KPSS_p_affichage","spec"]]


### Lecture scientifique
La série observée est identique dans les deux lignes. Pourtant, la conclusion change radicalement parce que la composante déterministe admise dans le test change. **Une série stationnaire autour d'une tendance déterministe n'est pas une série à racine unitaire.**


## 8. Mini-laboratoire étudiant

1. Simulez un AR(1) avec $\phi=0{,}98$ en utilisant une graine fixe. Comparez sa trajectoire et son ACF à la marche aléatoire.
2. Comparez `ar1_phi_075` et `ar1_phi_neg_065`. Expliquez l'alternance des autocorrélations.
3. Testez ADF et KPSS sur `ma1_theta_070` et formatez correctement les p-values.
4. Différenciez une série déjà stationnaire et interprétez l'ACF obtenue.
5. Reproduisez l'expérience `c` versus `ct` avec une pente déterministe différente.
6. Expliquez en une phrase pourquoi « stationnaire » ne signifie pas « profitable à trader ».


In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4

# Q5

# Q6 : réponse argumentée dans une cellule Markdown.


## Checklist de maîtrise

Avant de passer au Notebook 2 :
- Je peux expliquer la stationnarité faible sans réciter une formule.
- Je peux distinguer bruit blanc faible, bruit blanc gaussien et marche aléatoire.
- Je peux expliquer ACF et PACF sans en faire des signatures infaillibles.
- Je peux interpréter ADF et KPSS sans inverser leurs hypothèses nulles.
- Je sais pourquoi `c` versus `ct` fait partie du résultat du test.
- Je sais qu'une p-value extrêmement petite n'est pas zéro et qu'une p-value KPSS peut être seulement bornée.
- Je peux expliquer pourquoi différencier une série déjà stationnaire peut être nuisible.
